# Agentic AI Data Analysis Agent using LangGraph

### Overview
This project implements a fully autonomous **Agentic AI workflow** for exploratory data analysis using **LangGraph**. This system models a stateful, directed acyclic graph (DAG) to simulate the workflow of a Data Scientist. The workflow combines LLM reasoning, dynamic code execution, and graph-based orchestration to simulate an end-to-end analytical process with minimal manual intervention.

The agent autonomously profiles raw data, architects an exploratory data analysis (EDA) strategy, generates and executes Python code, and synthesizes technical findings into an business-ready report.

---

## Logic & Flow Architecture
The system is built on a **Stateful Graph** where a shared memory object (State) is mutated by specialized nodes:

1.  **Dataset Profiler:** Performs initial ingestion and metadata extraction.
2.  **Strategic Planner:** Evaluates the profile to prioritize high-impact analytical paths.
3.  **Code Execution Engine:** An LLM-driven node that writes and runs sandboxed Python code to perform EDA.
4.  **Insight Interpreter:** Transforms raw computational output into human-readable technical insights.
5.  **Technical Reporting:** Generates the final multi-section analytical deliverable.

## Core Stack
* **Orchestration:** LangGraph (Stateful Workflows)
* **Compute Engine:** Groq (LLaMA 3.1 70B for high-reasoning code generation)
* **Processing:** Pandas, NumPy
* **Visualization:** Seaborn, Matplotlib (Agg Backend)

In [1]:
# Installing core orchestration and inference libraries
!pip install -U -q langgraph langchain langchain-groq pandas matplotlib seaborn groq

## Environment & Model Configuration

The workflow uses Groq-hosted LLaMA 3.3 70B as the reasoning engine for:
- analytical planning,
- code generation,
- result interpretation,
- and report synthesis.

In [2]:
import os
import sys
import warnings
import io
import traceback
from typing import TypedDict, Annotated, List

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from IPython.display import Markdown, display

# Setting non-interactive backend for automated plotting within the agent nodes
matplotlib.use('Agg')
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [ ]:
# API Key Configuration
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# Inference Engine: LLaMA 3. 70B
# Temperature 0.2 ensures deterministic code generation and structural reliability
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    max_tokens=3000
)

## Dataset Loading

The workflow uses the IBM Telco Customer Churn dataset.

This dataset contains customer demographics, account information, service usage details, and churn labels commonly used for retention analytics and predictive modeling tasks.

In [25]:
df = pd.read_csv('https://raw.githubusercontent.com/ArchanaAnalytics/DS-Portfolio/refs/heads/main/Datasets/customer_churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [26]:
df.shape

(7043, 21)

## Shared Workflow State

LangGraph workflows operate using a shared state object.

Each node:
- receives the current state,
- enriches it with new information,
- and forwards the updated context to downstream nodes.

In [4]:
class PipelineState(TypedDict):
    """
    Shared state object representing the agent's 'working memory'.
    Facilitates context persistence across graph nodes.
    """
    raw_profile      : str    # Metadata, dtypes, and sample stats
    execution_plan   : str    # LLM-derived analytical strategy
    generated_code   : str    # Python code for EDA execution
    terminal_output  : str    # Stdout captured from the code execution node
    analysis_insights: str    # Interpreted results
    executive_report : str    # Final Markdown deliverable
    exception_log    : str    # Tracebacks for debugging/retry logic

## Workflow Nodes

Each node represents a specialized stage within the autonomous analysis pipeline.

### Node 1 : Dataset Profiling

In [19]:
def profiler_node(state: PipelineState) -> PipelineState:
    """Node 1: Extracts high-density metadata for LLM context."""

    print("\n--- 🔍 Node 1: Dataset Profiling ---")

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    profile = f"""
    Dataset Shape: {df.shape}

    Columns:
    {list(df.columns)}

    Data Types:
    {df.dtypes.to_string()}

    Missing Values:
    {missing.to_string() if len(missing) > 0 else 'None'}

    Churn Distribution:
    {df['Churn'].value_counts(normalize=True).mul(100).round(2).to_string()}

    Sample Rows:
    {df.head(3).to_string()}
    """
    return {"raw_profile": profile}

### Node 2 : Analysis Planning

In [20]:
def strategic_planner_node(state: PipelineState) -> PipelineState:
    """Node 2: Determines analytical priorities based on the data profile."""

    print("\n--- 🗺️ Node 2: Planning Analysis ---")

    system_prompt = """
    You are a senior data analyst. Create a focused EDA strategy for a telecom churn dataset.
    Include churn analysis, customer segmentation, numerical trends, categorical comparisons,
    and useful visualizations using actual dataset columns.
    """

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Dataset Profile:\n{state['raw_profile']}")
    ])

    return {"execution_plan": response.content}

### Node 3 : Code Generation & Execution

In [21]:
def code_engine_node(state: PipelineState) -> PipelineState:
    """Node 3: Dynamic code generation and sandboxed execution."""

    print("\n--- 💻 Node 3: EDA Code Generation ---")

    system_prompt = """
    Generate concise executable Python EDA code using df, pandas, matplotlib, seaborn.
    Use print() for findings, save plots using plt.savefig(), avoid hallucinated columns,
    and return only clean Python code.
    """

    user_prompt = f"""
    Analysis Strategy:
    {state['execution_plan']}

    Focus on:
    - Churn distribution
    - Contract vs Churn
    - PaymentMethod vs Churn
    - InternetService vs Churn
    - tenure analysis
    - MonthlyCharges analysis
    """

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])

    generated_code = response.content.replace("```python", "").replace("```", "").strip()

    stdout_buffer, error_log = io.StringIO(), ""
    sys.stdout = stdout_buffer

    try:
        exec_env = {"df": df.copy(), "pd": pd, "np": np, "plt": plt, "sns": sns}
        exec(generated_code, exec_env)

    except Exception:
        error_log = traceback.format_exc()

    finally:
        sys.stdout = sys.__stdout__

    return {
    "generated_code": generated_code,
    "terminal_output": stdout_buffer.getvalue(),
    "exception_log": error_log
}

### Node 4 : Insight Interpretation

In [22]:
def insight_interpreter_node(state: PipelineState) -> PipelineState:
    """Node 4: Translates computational outputs into business intelligence."""

    print("\n--- 🔎 Node 4: Interpreting Findings ---")

    system_prompt = """
    You are a telecom business analyst. Based ONLY on the provided EDA output,
    identify major churn drivers, risky customer segments, behavioral patterns,
    and practical business recommendations.
    """

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"EDA Output:\n{state['terminal_output']}")
    ])

    return {"analysis_insights": response.content}

### Node 5 : Final Report Generation

In [23]:
def reporting_node(state: PipelineState) -> PipelineState:
    """Node 5: Compiles the final executive-ready deliverable."""

    print("\n--- 📄 Node 5: Generating Final Report ---")

    system_prompt = """
    Generate a concise professional markdown report with:
    Executive Summary, Dataset Overview, Key Findings,
    Churn Drivers, Recommendations, and Conclusion.
    Keep the report realistic and grounded in the provided insights.
    """

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"Insights:\n{state['analysis_insights']}")
    ])

    return {"executive_report": response.content}

## LangGraph Workflow Orchestration

The workflow is implemented as a sequential LangGraph pipeline where each node contributes new context to the shared state object.

In [44]:
# Initializing the State Machine
workflow_builder = StateGraph(PipelineState)

# Node Registration
workflow_builder.add_node("data_profiler", profiler_node)
workflow_builder.add_node("strategic_planner", strategic_planner_node)
workflow_builder.add_node("code_engine", code_engine_node)
workflow_builder.add_node("insight_interpreter", insight_interpreter_node)
workflow_builder.add_node("report_generator", reporting_node)

# Defining DAG Edges
workflow_builder.set_entry_point("data_profiler")
workflow_builder.add_edge("data_profiler", "strategic_planner")
workflow_builder.add_edge("strategic_planner", "code_engine")
workflow_builder.add_edge("code_engine", "insight_interpreter")
workflow_builder.add_edge("insight_interpreter", "report_generator")
workflow_builder.add_edge("report_generator", END)

# Finalizing the Agent
autonomous_agent = workflow_builder.compile()

## Execution & Validation

In [45]:
# Initial State Configuration
initial_state: PipelineState = {
    "raw_profile": "",
    "execution_plan": "",
    "generated_code": "",
    "terminal_output": "",
    "analysis_insights": "",
    "executive_report": "",
    "exception_log": ""
}

# Invoke the Autonomous Agent
final_state = autonomous_agent.invoke(initial_state)

## Final Report

In [47]:
display(Markdown(final_state["executive_report"]))

# Executive Summary
This report provides an analysis of customer churn drivers and behavior patterns based on the provided Exploratory Data Analysis (EDA) output. The key findings highlight the significant impact of internet service, payment method, and contract type on customer churn. The report also identifies risky customer segments and provides practical recommendations to reduce churn and improve customer retention.

# Dataset Overview
The dataset consists of customer information, including internet service, payment method, contract type, tenure, and monthly charges. The analysis is based on a total of 7043 customers, with 1869 churning and 5174 remaining active.

# Key Findings
* Customers with Fiber optic internet service have a significantly higher churn rate (41.8%) compared to those with DSL (18.9%) and No internet service (7.4%).
* Electronic check payment method is associated with a higher churn rate (45.3%) compared to other payment methods.
* Month-to-month contracts have a higher churn rate (42.7%) compared to One year and Two year contracts.
* Customers with higher tenure tend to have a lower churn rate, with a mean tenure of 32.37 months.
* Customers with higher monthly charges may be more likely to churn, with a mean monthly charge of $64.76.

# Churn Drivers
The major churn drivers identified in this analysis are:
1. **Internet Service**: Fiber optic internet service is associated with a higher churn rate.
2. **Payment Method**: Electronic check payment method is associated with a higher churn rate.
3. **Contract Type**: Month-to-month contracts are associated with a higher churn rate.

# Recommendations
To reduce customer churn and improve retention, the following recommendations are made:
1. **Offer incentives or promotions** to customers with Fiber optic internet service.
2. **Consider alternative payment methods** for customers who use electronic checks.
3. **Develop targeted retention strategies** for customers with Month-to-month contracts.
4. **Analyze the relationship** between tenure, monthly charges, and churn to identify potential opportunities for improvement.

# Conclusion
This report provides valuable insights into customer churn drivers and behavior patterns. By implementing the recommended strategies, the company can reduce churn, improve customer retention, and increase revenue. Further analysis and validation are necessary to ensure the effectiveness of these recommendations.

---
## Observations

The generated analysis successfully identified meaningful churn patterns directly from raw tabular data without manually predefined analytical rules.

The workflow autonomously:
- detected high-risk customer segments,
- identified churn-correlated behavioral attributes,
- generated executable EDA logic,
- and synthesized business-oriented recommendations from runtime outputs.

The resulting reports remained grounded in observable dataset statistics rather than hardcoded domain assumptions, demonstrating the effectiveness of graph-based agent orchestration for autonomous analytical workflows.